In [0]:
%run "../includes/configuration"

In [0]:
movies_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/movies")
).select("movie_id", "budget", "revenue", "year_release_date")

In [0]:
countries_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/countries")
).select("country_id", "country_name")

In [0]:
production_countries_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/production_countries")
)


In [0]:
from pyspark.sql.functions import col, sum, avg, rank, dense_rank, row_number
from pyspark.sql.window import Window

In [0]:
rank_budget_revenue = (
    Window.partitionBy("year_release_date").
        orderBy(
            col("total_budget").desc(), col("total_revenue").desc()
            )
)

results_group_movie_country = (
    movies_df.filter(movies_df.year_release_date >= 2015)
        .join(production_countries_df,
              movies_df["movie_id"] == production_countries_df["movie_id"], "inner")
        .join(countries_df,
              production_countries_df["country_id"] == countries_df["country_id"], "inner")
        .select(
            "year_release_date",
            "country_name",
            "budget",
            "revenue"
        ).groupBy("year_release_date", "country_name")
            .agg(
                sum("budget").alias("total_budget"),
                sum("revenue").alias("total_revenue")
        )
        .orderBy(col("total_budget").desc())
        .withColumn("budget_rank",
                    row_number()
                        .over(Window.partitionBy("year_release_date")
                              .orderBy(col("total_budget").desc())))
        .withColumn("rank_budget_revenue",
                    dense_rank().over(rank_budget_revenue))
)
display(results_group_movie_country)

In [0]:
(
    results_group_movie_country
        .write
            .format("parquet")
            .mode("overwrite")
            .save(f"{gold_folder_path}/results_group_movie_country")
)